In [60]:
import sys, os
print(sys.executable)
print(os.getcwd())

/Users/gizemtotkanli/.pyenv/versions/workintech_current/bin/python
/Users/gizemtotkanli/code/totkanligizem/03-Decision-Science/03-Logistic-Regression/data-logit


In [61]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [62]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [63]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [64]:
features = [
    "wait_time",
    "delay_vs_expected",
    "number_of_items",
    "number_of_sellers",
    "price",
    "freight_value",
    "distance_seller_customer",
    "expected_wait_time",
]

🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [65]:
orders.columns.tolist()

['delay_vs_expected',
 'dim_is_five_star',
 'dim_is_one_star',
 'expected_wait_time',
 'freight_value',
 'number_of_items',
 'number_of_sellers',
 'order_id',
 'order_status',
 'price',
 'review_score',
 'wait_time']

In [66]:
candidate_features = [
    "wait_time",
    "delay_vs_expected",
    "number_of_items",
    "number_of_sellers",
    "price",
    "freight_value",
    "expected_wait_time",
    "distance_seller_customer",  # varsa al, yoksa otomatik düşecek
]

features = [f for f in candidate_features if f in orders.columns]
features

['wait_time',
 'delay_vs_expected',
 'number_of_items',
 'number_of_sellers',
 'price',
 'freight_value',
 'expected_wait_time']

In [67]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

X = orders[features].copy()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=features,
    index=orders.index
)

orders_standardized = orders.copy()
orders_standardized[features] = X_scaled

orders_standardized[features].describe().T[["mean", "std"]]

,mean,std
wait_time,6.818459e-17,1.000005
delay_vs_expected,-2.401283e-17,1.000005
number_of_items,-1.792069e-16,1.000005
number_of_sellers,2.736277e-16,1.000005
price,1.348869e-17,1.000005
freight_value,-9.190097e-18,1.000005
expected_wait_time,6.548685e-16,1.000005


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [68]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd

# VIF için sabit terim (intercept) ekleyelim
X_vif = sm.add_constant(orders_standardized[features])

vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

vif.sort_values("VIF", ascending=False)

,feature,VIF
1,wait_time,2.832123
2,delay_vs_expected,2.378744
6,freight_value,1.576585
7,expected_wait_time,1.443791
3,number_of_items,1.347989
5,price,1.204976
4,number_of_sellers,1.095545
0,const,1.000000


In [69]:
orders_standardized["dim_is_one_star"] = orders_standardized["review_score"] == 1
orders_standardized["dim_is_five_star"] = orders_standardized["review_score"] == 5

In [70]:
orders_standardized[["dim_is_one_star", "dim_is_five_star"]].mean()

dim_is_one_star     0.097651
dim_is_five_star    0.592112
dtype: float64

## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [71]:
# 1) Kolon var mı, kaç tane var?
[c for c in orders_standardized.columns if "dim_is_one_star" in c]

['dim_is_one_star']

In [72]:
orders_standardized["dim_is_one_star"] = (orders_standardized["review_score"] == 1).astype(int)
orders_standardized["dim_is_five_star"] = (orders_standardized["review_score"] == 5).astype(int)

orders_standardized[["dim_is_one_star", "dim_is_five_star"]].head()

,dim_is_one_star,dim_is_five_star
0,0,0
1,0,0
2,0,1
3,0,1
4,0,1


In [73]:
orders_standardized["dim_is_one_star"].value_counts(dropna=False)

dim_is_one_star
0    86510
1     9362
Name: count, dtype: int64

In [74]:
logit_one = smf.logit(
    formula="dim_is_one_star ~ " + " + ".join(features),
    data=orders_standardized
).fit(disp=False)

logit_one.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95864
Method:                           MLE   Df Model:                            7
Date:                Sun, 04 Jan 2026   Pseudo R-squ.:                  0.1460
Time:                        23:16:06   Log-Likelihood:                -26191.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -2.4811      0.013   -189.140      0.000      -2.507      -2.455
wait_time              0.7931      0.019     41.309      0.000       0.755       0.831
delay_vs_expected      0.1614      0.020      7.955      0.000       0.122       0.201
number_of_items        0.2635      0.011     24.854      0.000       0.243       0.284
number_of_sellers      0.1862      0.008     23.511      0.000       0.171       0.202
price                  0.0567      0.011      5.000      0.000       0.034       0.079
freight_value         -0.0404      0.013     -3.067      0.002      -0.066      -0.015
expected_wait_time    -0.2473      0.016    -15.357      0.000      -0.279      -0.216
======================================================================================
"""

`Logit 5️⃣`

In [75]:
logit_five = smf.logit(
    formula="dim_is_five_star ~ " + " + ".join(features),
    data=orders_standardized
).fit(disp=False)

logit_five.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       dim_is_five_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95864
Method:                           MLE   Df Model:                            7
Date:                Sun, 04 Jan 2026   Pseudo R-squ.:                 0.05816
Time:                        23:16:06   Log-Likelihood:                -61047.
converged:                       True   LL-Null:                       -64817.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3413      0.007     47.715      0.000       0.327       0.355
wait_time             -0.5386      0.012    -43.948      0.000      -0.563      -0.515
delay_vs_expected     -0.3925      0.024    -16.284      0.000      -0.440      -0.345
number_of_items       -0.1439      0.008    -17.320      0.000      -0.160      -0.128
number_of_sellers     -0.1456      0.008    -18.553      0.000      -0.161      -0.130
price                  0.0173      0.008      2.272      0.023       0.002       0.032
freight_value          0.0177      0.009      2.039      0.041       0.001       0.035
expected_wait_time     0.0939      0.008     11.138      0.000       0.077       0.110
======================================================================================
"""

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [76]:
a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = []

# Katsayıların mutlak değeriyle kıyas
coef_one = logit_one.params
coef_five = logit_five.params

if abs(coef_five["delay_vs_expected"]) > abs(coef_one["delay_vs_expected"]):
    your_answer.append(a)

if abs(coef_five["wait_time"]) > abs(coef_one["wait_time"]):
    your_answer.append(b)

your_answer

['delay_vs_expected influences five_star ratings even more than one_star ratings']

In [77]:
a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

Logistic regression results show that delivery delays (delay_vs_expected) reduce the probability of receiving 5-star reviews more strongly than they increase the probability of 1-star reviews.
This suggests that customers who are generally satisfied are more sensitive to delivery issues when deciding whether to give a perfect rating, whereas extremely negative reviews are driven by additional factors beyond delivery performance alone.


🧪 __Kodunu Test Et__

In [78]:
from nbresult import ChallengeResult

result = ChallengeResult(
    "logit",
    answers=your_answer
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/gizemtotkanli/.pyenv/versions/workintech_current/bin/python
cachedir: .pytest_cache
rootdir: /Users/gizemtotkanli/code/totkanligizem/03-Decision-Science/03-Logistic-Regression/data-logit/tests
plugins: dash-3.3.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.00s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!